# 05 · Regras de associação

**CRISP-DM fase 4** · Apriori, Eclat e FP-Growth

A terceira pergunta da diretoria: *que padrões de coocorrência existem nos dados que
sustentem ou contradigam a segmentação?*

In [ ]:
import sys, warnings
from pathlib import Path

# O notebook roda a partir de `notebooks/`; a raiz do projeto entra no path
# para que `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from src import config
from src.utils.io import configurar_estilo

configurar_estilo()
print(f"Projeto em {RAIZ}")

In [ ]:
from src.model import associacao as assoc
from src.model import personas as pers

segmentada = pd.read_parquet(config.SEGMENTOS)
segmentos = segmentada["segmento"].to_numpy()
segmentada["codigo_segmento"].value_counts()

## Preparação transacional

A entidade da transação é o **imóvel**, não o bairro nem o ano: a pergunta de negócio
é sobre a composição do produto. Agrupar por bairro juntaria imóveis distintos em uma
cesta só e inflaria o suporte artificialmente.

Os itens vêm de quatro origens, e só uma delas exige discretização:

| Origem | Discretização |
|---|---|
| Nominais | nenhuma — a categoria já é o item |
| Ordinais | faixas de mercado (cortes em 4 e 7 na escala 1–10) |
| Numéricas | quartis da própria carteira |
| Binárias construtivas | nenhuma — presença do item |

In [ ]:
transacoes = assoc.montar_transacoes(segmentada, segmentos)
print(f"{transacoes.shape[0]} transações × {transacoes.shape[1]} itens")
transacoes.mean().sort_values(ascending=False).head(15).to_frame("suporte").round(3)

## Calibrar o limiar **é** o exercício

Suporte alto devolve apenas o óbvio; suporte baixo explode combinatoriamente. A tabela
abaixo torna visível onde fica o ponto de operação.

In [ ]:
from src.pipeline.regras import sensibilidade_limiares

sem_segmento = transacoes[[c for c in transacoes.columns if not c.startswith("segmento:")]]
sensibilidade_limiares(sem_segmento)

Ponto de operação escolhido: **suporte ≥ 10%** (293 imóveis), **até 3 itens**,
confiança ≥ 70%, lift ≥ 1,30.

A 5% e 4 itens seriam 6.274 regras — volume que nenhuma área comercial lê, e que não
traz padrão de negócio novo, apenas variações redundantes das mesmas combinações.

## Os três algoritmos

Apriori, Eclat e FP-Growth chegam ao **mesmo** resultado por caminhos diferentes. O
Eclat aqui é implementação própria: em vez da lista de itens de cada transação
(representação horizontal), guarda para cada item a lista de transações em que ele
aparece — o *tidset*. O suporte de um conjunto vira o tamanho da interseção dos
tidsets, uma operação de conjunto, sem varrer a base de novo.

In [ ]:
comparativo, frequentes = assoc.comparar_algoritmos(sem_segmento, 0.10, max_tamanho=3)
comparativo

In [ ]:
assoc.itens_por_tamanho(frequentes)

In [ ]:
assoc.combinacoes_frequentes(frequentes, tamanho=3, quantas=10)

## Geração e poda

Quatro filtros, em ordem. Cada um remove um tipo diferente de lixo.

In [ ]:
brutas = assoc.gerar_regras(frequentes, confianca_minima=0.70, lift_minimo=1.30)
sem_triviais = assoc.remover_regras_triviais(brutas)
sem_redundantes = assoc.podar_redundantes(sem_triviais)
podadas, tautologicas = assoc.remover_tautologias(sem_redundantes)

pd.DataFrame(
    [
        {"etapa": "Conjuntos frequentes", "regras": len(frequentes)},
        {"etapa": "Confiança ≥ 70% e lift ≥ 1,30", "regras": len(brutas)},
        {"etapa": "Sem faixas da mesma variável", "regras": len(sem_triviais)},
        {"etapa": "Sem redundância por especialização", "regras": len(sem_redundantes)},
        {"etapa": "Sem tautologia semântica", "regras": len(podadas)},
    ]
).set_index("etapa")

### O erro clássico: confiança alta com lift 1

Sem o filtro de lift, estas regras entrariam no relatório como descobertas. Elas dizem
apenas que o consequente é comum — valeriam para qualquer antecedente.

In [ ]:
sem_filtro = assoc.gerar_regras(frequentes, confianca_minima=0.70, lift_minimo=0.0)
assoc.formatar_regras(sem_filtro[sem_filtro["lift"] <= 1.0].nlargest(8, "confidence"), 8)

### A tautologia que nenhuma métrica pega

Uma casa de estilo `2Story` tem segundo pavimento porque é isso que `2Story`
significa. Suporte, confiança e lift são altos **justamente** porque a relação é
perfeita.

Só o conhecimento do domínio separa a tautologia da descoberta — e por isso a lista de
pares definicionais é explícita em `src/model/associacao.py`, e não inferida.

In [ ]:
assoc.formatar_regras(tautologicas, 8)

## As regras que sobreviveram

In [ ]:
assoc.formatar_regras(podadas, 20)

In [ ]:
fig, eixo = plt.subplots(figsize=(9, 5), constrained_layout=True)
dispersao = eixo.scatter(podadas["support"], podadas["confidence"], c=podadas["lift"],
                         cmap="viridis", s=40, alpha=0.75, edgecolors="white", linewidths=0.4)
fig.colorbar(dispersao, ax=eixo, label="lift")
eixo.set_xlabel("suporte")
eixo.set_ylabel("confiança")
eixo.set_title("Mapa das regras podadas")
plt.show()

## Cruzamento com a segmentação

É o cruzamento que valida ou contesta a segmentação: se as regras de cada grupo forem
as mesmas da base toda, o agrupamento não achou nada que a frequência global já não
dissesse.

In [ ]:
por_segmento = assoc.regras_por_segmento(transacoes, segmentos, 0.20, 0.70, 1.30, 3)
por_segmento.groupby("segmento").size().rename("regras válidas no segmento").to_frame()

In [ ]:
diferencial = assoc.lift_diferencial(por_segmento, podadas, minimo_diferenca=0.20)
diferencial["persona"] = diferencial["segmento"].map(lambda s: pers.PERSONAS[int(s)]["codigo"])
diferencial.groupby("persona").head(5)[
    ["persona", "regra", "suporte_no_segmento", "lift_no_segmento", "lift_na_base", "diferenca"]
]

### Leitura de negócio

Os padrões diferenciais confirmam as personas construídas na clusterização, por um
caminho **independente** dela:

- **S1 (Requalificação sem porão)** — `Duplex` aparece junto de dois banheiros e
  conservação média com lift acima de 2,4, e essa combinação **não existe** na base
  inteira. É a assinatura do estoque de renda.
- **S2 (Compacto de entrada)** — zoneamento `RM` (média densidade) leva a terreno no
  primeiro quartil. Lote pequeno em zona adensada é o produto de entrada.
- **S3 (Familiar consolidado)** — garagem destacada leva a imóvel no quartil mais
  velho com segundo pavimento: o casario consolidado dos bairros antigos.
- **S4 (Novo de alto padrão)** — sobrado com lareira leva a área no quartil superior.

Duas metades do projeto que se sustentam mutuamente.